In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.metrics import make_scorer, f1_score, roc_auc_score, precision_score, balanced_accuracy_score, accuracy_score, recall_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN

from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, export_graphviz
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from catboost import CatBoostClassifier
from sklearn.svm import SVC


In [ ]:
# define constants according to your dataset

DATASET = "sample.csv"
COLUMN_NAME_LOS = "DurationHospStayDay"
COLUMN_NAME_ADMISSION_DATE = "DateAdmission"
COLUMN_NAME_TARGET = "Outcome"

In [ ]:
df = pd.read_csv('../data/' + DATASET, low_memory=False)
df = df.drop(columns=[COLUMN_NAME_LOS])
df.drop(COLUMN_NAME_ADMISSION_DATE, axis=1, inplace=True)

In [ ]:
# Example: binary outcome
X = df.drop(COLUMN_NAME_TARGET, axis=1)
y = df[COLUMN_NAME_TARGET]

In [ ]:
label_encoder = LabelEncoder()
label_mapping = {}
for column in X.columns:
  if X[column].dtype == 'object':
    X[column] = X[column].astype(str)
    X[column] = label_encoder.fit_transform(X[column])
    label_mapping[column] = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

print(label_mapping.keys())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=123
)

In [ ]:
scoring = {
    "f1": make_scorer(f1_score, average="binary", pos_label="Dead"),
    "roc_auc": make_scorer(roc_auc_score, needs_proba=True),
    "precision": make_scorer(precision_score, pos_label="Dead"),
    "accuracy": make_scorer(accuracy_score),
    "balanced_accuracy": make_scorer(balanced_accuracy_score),
    "recall": make_scorer(recall_score, pos_label="Dead")
}

In [ ]:
models = [{
        "model": LogisticRegression(max_iter=3000),
        "model_name": "Logistic Regression",
        "hyperparameters": {
            "model__C": [0.01, 0.1, 1, 10, 100],   # regularization strength
            "model__penalty": ["l1", "l2", "elasticnet"],
            "model__l1_ratio": [0, 0.5, 1]         # only used if elasticnet
        }
    },
    {
        "model": GaussianNB(),
        "model_name": "Gaussian Naive Bayes",
        "hyperparameters": {
            "model__var_smoothing": [1e-9, 1e-8, 1e-7, 1e-6]   # handles numerical stability
        }
    },
    {
        "model": DecisionTreeClassifier(),
        "model_name": "Decision Tree Classifier",
        "hyperparameters": {
            "model__criterion": ["gini", "entropy", "log_loss"],
            "model__max_depth": [None, 5, 10, 20, 50],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": [None, "sqrt", "log2"]
        }
    },
    {
        "model": RandomForestClassifier(),
        "model_name": "Random Forest Classifier",
        "hyperparameters": {
            "model__n_estimators": [100, 200, 500],
            "model__criterion": ["gini", "entropy", "log_loss"],
            "model__max_depth": [None, 10, 20, 50],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": ["sqrt", "log2", None]
        }
    },
    {
        "model": SVC(probability=True),
        "model_name": "Support Vector Classifier",
        "hyperparameters": {
            "model__C": [0.01, 0.1, 1, 10, 100],
            "model__kernel": ["linear", "rbf", "poly"],
            "model__degree": [2, 3, 4],   # only for poly
            "model__gamma": ["scale", "auto"]
        }
    },
    {
        "model": ExtraTreesClassifier(criterion="entropy",random_state=123),
        "model_name": "Extra Trees Classifier",
        "hyperparameters": {
            "model__n_estimators": [100, 200, 500],
            "model__criterion": ["gini", "entropy", "log_loss"],
            "model__max_depth": [None, 10, 20, 50],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": ["sqrt", "log2", None]
        }
    },
    {
        "model": KNeighborsClassifier(n_neighbors=3,metric='minkowski',p=2),
        "model_name": "K-Neighbors Classifier",
        "hyperparameters": {
            "model__n_neighbors": [3, 5, 7, 11],
            "model__weights": ["uniform", "distance"],
            "model__metric": ["minkowski", "manhattan"],
            "model__p": [1, 2]   # 1=Manhattan, 2=Euclidean
        }
    },
    {
        "model": GradientBoostingClassifier(),
        "model_name": "Gradient Boosting Classifier",
        "hyperparameters": {
            "model__n_estimators": [100, 200, 500],
            "model__learning_rate": [0.01, 0.05, 0.1],
            "model__max_depth": [3, 5, 7],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
            "model__subsample": [0.6, 0.8, 1.0],
            "model__max_features": ["sqrt", "log2", None]
        }
    },
    {
        "model": CatBoostClassifier(verbose=False),
        "model_name": "CatBoost Classifier",
        "hyperparameters": {
            "model__iterations": [200, 500, 1000],
            "model__learning_rate": [0.01, 0.05, 0.1],
            "model__depth": [4, 6, 8, 10],
            "model__l2_leaf_reg": [1, 3, 5, 7]
        }
    }]

In [ ]:
models = [
    {
        "model": DecisionTreeClassifier(),
        "model_name": "Decision Tree Classifier",
        "hyperparameters": {
            "model__criterion": ["gini", "entropy", "log_loss"],
            "model__max_depth": [None, 5, 10, 20, 50],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": [None, "sqrt", "log2"]
        }
    },
    {
        "model": RandomForestClassifier(),
        "model_name": "Random Forest Classifier",
        "hyperparameters": {
            "model__n_estimators": [100, 200, 500],
            "model__criterion": ["gini", "entropy", "log_loss"],
            "model__max_depth": [None, 10, 20, 50],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": ["sqrt", "log2", None]
        }
    },
    {
        "model": ExtraTreesClassifier(criterion="entropy",random_state=123),
        "model_name": "Extra Trees Classifier",
        "hyperparameters": {
            "model__n_estimators": [100, 200, 500],
            "model__criterion": ["gini", "entropy", "log_loss"],
            "model__max_depth": [None, 10, 20, 50],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": ["sqrt", "log2", None]
        }
    },
    {
        "model": GradientBoostingClassifier(),
        "model_name": "Gradient Boosting Classifier",
        "hyperparameters": {
            "model__n_estimators": [100, 200, 500],
            "model__learning_rate": [0.01, 0.05, 0.1],
            "model__max_depth": [3, 5, 7],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
            "model__subsample": [0.6, 0.8, 1.0],
            "model__max_features": ["sqrt", "log2", None]
        }
    },
    {
        "model": CatBoostClassifier(verbose=False),
        "model_name": "CatBoost Classifier",
        "hyperparameters": {
            "model__iterations": [200, 500, 1000],
            "model__learning_rate": [0.01, 0.05, 0.1],
            "model__depth": [4, 6, 8, 10],
            "model__l2_leaf_reg": [1, 3, 5, 7]
        }
    }]

## Sampling Selection

In [ ]:
resamplers = {
    "None": None,
    "SMOTE": SMOTE(random_state=123),
    # "RandomUnder": RandomUnderSampler(random_state=123),
    "SMOTEENN": SMOTEENN(random_state=123),
    "ROS": RandomOverSampler(random_state=123),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

model_resampler_results = []

for name, sampler in resamplers.items():
    
    for model in models:
        print(f"Evaluating {model['model_name']} with sampler {name}")
        model_resampler_result = {}
        model_resampler_results.append(model_resampler_result)
        model_resampler_result["model"] = model["model_name"]
        model_resampler_result["sampler"] = name

        steps = []
        if sampler is not None:
            steps.append(("sampler", sampler))
        steps.append(("model", model["model"]))
        
        pipeline = ImbPipeline(steps=steps)
        
        for scoring_name in scoring.keys():
            scores = cross_val_score(
                pipeline, X_train, y_train, cv=cv, scoring=scoring[scoring_name]
            )
            model_resampler_result[scoring_name] = np.mean(scores)

model_resampler_results_df = pd.DataFrame(model_resampler_results)
model_resampler_results_df.round(4)
display(model_resampler_results_df)

In [ ]:
model_resampler_results_df.to_csv("model_resampler_results.csv")

In [ ]:
tmp = model_resampler_results_df.copy(deep=True)\
    .drop(columns=["model", "f1", "precision", "accuracy", "balanced_accuracy", "recall"])\
    .groupby(by=["sampler"]).mean()\
    .sort_values(by=["roc_auc"], ascending=False)
# tmp = model_resampler_results_df.copy(deep=True)\
#     .drop(columns=["model", "f1", "precision", "accuracy", "roc_auc", "recall"])\
#     .groupby(by=["sampler"]).mean()\
#     .sort_values(by=["balanced_accuracy"], ascending=False)
tmp

## Hyperparameter tuning

In [ ]:
best_hyperparameters_by_models = []
for model in models:
    print(f"Evaluating {model['model_name']} with hyperparameter tuning")
    best_hyperparameters = {}
    best_hyperparameters_by_models.append(best_hyperparameters)
    best_hyperparameters["model"] = model["model_name"]
    
    pipeline = ImbPipeline([
        ("sampler", RandomOverSampler(random_state=123)),
        ("model", model["model"])
    ])

    param_grid = model["hyperparameters"]

    grid = GridSearchCV(
        pipeline,
        param_grid,
        cv=cv,
        scoring="balanced_accuracy",
        n_jobs=-1,
        verbose=2
    )    
    grid.fit(X_train, y_train)
    
    best_hyperparameters["best_balanced_accuracy"] = grid.best_score_
    best_hyperparameters["best_params"] = grid.best_params_
    print(f"{model['model_name']}: best balanced_accuracy {grid.best_score_:.3f} with {grid.best_params_}")

In [ ]:
best_hyperparameters_by_models

In [ ]:
best_hyperparameters_json = [{'model': 'Logistic Regression',
  'best_balanced_accuracy': 0.8876506364922208,
  'best_params': {'model__C': 1,
   'model__l1_ratio': 0,
   'model__penalty': 'l2'}},
 {'model': 'Gaussian Naive Bayes',
  'best_balanced_accuracy': 0.8750848656294201,
  'best_params': {'model__var_smoothing': 1e-07}},
 {'model': 'Decision Tree Classifier',
  'best_balanced_accuracy': 0.9049363507779349,
  'best_params': {'model__criterion': 'log_loss',
   'model__max_depth': None,
   'model__max_features': 'log2',
   'model__min_samples_leaf': 2,
   'model__min_samples_split': 5}},
 {'model': 'Random Forest Classifier',
  'best_balanced_accuracy': 0.915642857142857,
  'best_params': {'model__criterion': 'log_loss',
   'model__max_depth': 10,
   'model__max_features': 'log2',
   'model__min_samples_leaf': 4,
   'model__min_samples_split': 10,
   'model__n_estimators': 100}},
 {'model': 'Support Vector Classifier',
  'best_balanced_accuracy': 0.8807001414427157,
  'best_params': {'model__C': 0.1,
   'model__degree': 2,
   'model__gamma': 'scale',
   'model__kernel': 'linear'}},
 {'model': 'Extra Trees Classifier',
  'best_balanced_accuracy': 0.8995813295615276,
  'best_params': {'model__criterion': 'entropy',
   'model__max_depth': None,
   'model__max_features': None,
   'model__min_samples_leaf': 4,
   'model__min_samples_split': 2,
   'model__n_estimators': 100}},
 {'model': 'K-Neighbors Classifier',
  'best_balanced_accuracy': 0.7257454031117397,
  'best_params': {'model__metric': 'minkowski',
   'model__n_neighbors': 11,
   'model__p': 1,
   'model__weights': 'uniform'}},
 {'model': 'Gradient Boosting Classifier',
  'best_balanced_accuracy': 0.9221725601131542,
  'best_params': {'model__learning_rate': 0.05,
   'model__max_depth': 3,
   'model__max_features': 'log2',
   'model__min_samples_leaf': 2,
   'model__min_samples_split': 5,
   'model__n_estimators': 100,
   'model__subsample': 0.8}},
 {'model': 'CatBoost Classifier',
  'best_balanced_accuracy': 0.8995714285714286,
  'best_params': {'model__depth': 10,
   'model__iterations': 200,
   'model__l2_leaf_reg': 7,
   'model__learning_rate': 0.05}}]

## Feature Selection

- Print out decision tree and analyse most common feature
- Sort by node level and occurence
- Brute force the combination to get best result

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier

clf = ExtraTreesClassifier(
    n_estimators=100,
    random_state=42
)
clf.fit(X, y)


In [ ]:
def explain_tree_path(tree, X_row, feature_names):
    """
    Generate a simple, human-friendly explanation of the decision path.
    tree: an individual tree inside ExtraTreesClassifier (clf.estimators_[i])
    X_row: a single-row DataFrame (X.iloc[[i]])
    feature_names: X.columns
    """

    tree_struct = tree.tree_
    export_graphviz(tree, out_file="ZDecision path Demo.dot", feature_names=feature_names, filled=True)
    x = X_row.iloc[0]

    node_indicator = tree.decision_path(X_row)
    leaf_id = tree.apply(X_row)[0]

    explanation = []
    node_index = node_indicator.indices

    for node_id in node_index:
        if node_id == leaf_id:
            explanation.append("➡️ Reached final decision (leaf node).\n")
            break

        feature_index = tree_struct.feature[node_id]
        threshold = tree_struct.threshold[node_id]

        feature_name = feature_names[feature_index]
        feature_value = x[feature_index]

        if feature_value <= threshold:
            decision = (
                f"Because **{feature_name} = {feature_value}**, which is **LOWER** "
                f"than the threshold {threshold:.2f}, the model followed the **YES** branch."
            )
        else:
            decision = (
                f"Because **{feature_name} = {feature_value}**, which is **HIGHER** "
                f"than the threshold {threshold:.2f}, the model followed the **NO** branch."
            )

        explanation.append(decision + "\n")

    pred = tree.predict(X_row)[0]
    explanation.append(f"🟩 Tree’s Prediction: {pred}")

    return "\n".join(explanation)


In [ ]:
row = X.iloc[[0]]
explanation = explain_tree_path(
    tree=clf.estimators_[0],
    X_row=row,
    feature_names=X.columns
)
print(explanation)

## Best Model

In [ ]:
best_model = GradientBoostingClassifier(max_depth=3, max_features='log2', min_samples_leaf=2, min_samples_split=5,n_estimators=100,subsample=0.8)
X_ros, y_ros = RandomOverSampler(random_state=123).fit_resample(X_train, y_train)
best_model.fit(X_ros, y_ros)

In [ ]:
y_pred = best_model.predict(X_test)
y_pred_proba = best_model.predict_proba(X_test)[:, 1]
res = {
    "f1" : f1_score(y_test, y_pred, average="binary", pos_label="Dead"),
    "roc_auc" : roc_auc_score(y_test, y_pred_proba),
    "precision" : precision_score(y_test, y_pred, pos_label="Dead"),
    "accuracy" : accuracy_score(y_test, y_pred),
    "balanced_accuracy" : balanced_accuracy_score(y_test, y_pred),
    "recall" : recall_score(y_test, y_pred, pos_label="Dead")
}
res

In [ ]:
from sklearn.tree import plot_tree, export_text
# inspect 1 tree
tree = best_model.estimators_[0][0]
# print(plot_tree(tree))
print(export_text(tree))

In [ ]:
from sklearn.tree import export_graphviz

for i, estimator_array in enumerate(best_model.estimators_):
    # In GradientBoostingClassifier, estimators_ is a 2D array where each row
    # represents a stage and each column represents a class (for multi-class)
    # For binary classification, there is only one tree per stage.
    tree = estimator_array[0] # Get the first (and only) tree for binary classification

    print(f"Decision paths for Tree {i+1}:")
    # You can use export_text to get a text-based representation of the tree rules
    tree_rules = export_graphviz(tree, out_file=f'Decision paths for Tree {i+1}.dot', feature_names=X_train.columns, class_names=best_model.classes_, filled=True)

In [ ]:
def getRealNodes(nodes):
    real_nodes = []
    for node in nodes:
        if node.get_attributes().get("label") is not None:
            real_nodes.append(node)
    return real_nodes

In [ ]:
import pydot
from collections import deque

def jsonify_nodes(root_node_name, node_list, edge_list):
    """
    Traverses a pydot graph from a root node and returns a list of dictionaries 
    with each node's label and level.

    Args:
        root_node_name (str): The name of the starting (root) node.
        node_list (list): A list of pydot Node objects.
        edge_list (list): A list of pydot Edge objects.

    Returns:
        list: A list of dictionaries, each containing a node's 'label' and 'level'.
    """
    # 1. Build an adjacency list for efficient neighbor lookup
    adj = {node.get_name(): [] for node in node_list}
    for edge in edge_list:
        source = edge.get_source()
        destination = edge.get_destination()
        if source in adj:
            adj[source].append(destination)
    
    # 2. Map node names to their labels
    # Use .get_label() but handle the case where it returns a name, which occurs
    # if no explicit label is set.
    name_to_label_map = {
        node.get_name(): node.get_attributes().get("label")
        if node.get_label() != node.get_name() else node.get_name()
        for node in node_list
    }

    # 3. Prepare data structures for BFS
    queue = deque([(root_node_name, 0)])  # Store tuples of (node_name, level)
    visited = {root_node_name}
    result_list = []

    # 4. Perform the BFS traversal
    while queue:
        current_node_name, level = queue.popleft()
        
        # Get the label from the map we created
        current_label = name_to_label_map.get(current_node_name, current_node_name)
        
        # Add the current node's label and level to the result
        result_list.append({
            "label": current_label,
            "level": level
        })

        # Enqueue all unvisited neighbors
        for neighbor_name in adj.get(current_node_name, []):
            if neighbor_name not in visited:
                visited.add(neighbor_name)
                queue.append((neighbor_name, level + 1))
    
    return result_list

In [ ]:
def tabulate_nodes(json_nodes):
    for node in json_nodes:
        node['label'] = node['label'].split(" ")[0].replace('"', '')
        
    return pd.DataFrame(json_nodes)

test_list = [{
    "label": "BW <= ashdjkasdhk"
}]

tabulate_nodes(test_list)

In [ ]:
graph = pydot.graph_from_dot_file('Decision paths for Tree 1.dot')
print(len(graph[0].get_nodes()))

real_nodes = getRealNodes(graph[0].get_nodes())
print("nodes")
for i, node in enumerate(real_nodes):
    # print(node)
    print(f"Node {i} attributes: {node.get_attributes()}")
    
print("edges")
for edge in graph[0].get_edges():
    print(edge)

In [ ]:
graph = pydot.graph_from_dot_file('Decision paths for Tree 1.dot')
real_nodes = getRealNodes(graph[0].get_nodes())

jsonified_nodes = jsonify_nodes(real_nodes[0].get_name(), graph[0].get_nodes(), graph[0].get_edges())
tabulate_nodes(jsonified_nodes)

In [ ]:
df_list = []
for i in range(100):
    graph = pydot.graph_from_dot_file(f'Decision paths for Tree {i+1}.dot')
    real_nodes = getRealNodes(graph[0].get_nodes())
    jsonified_nodes = jsonify_nodes(real_nodes[0].get_name(), graph[0].get_nodes(), graph[0].get_edges())
    df_list.append(tabulate_nodes(jsonified_nodes))
    
aggregated_df = pd.concat(df_list, ignore_index=True)

In [ ]:
sorted_df = aggregated_df.sort_values(by=['level', 'label'], ascending=[True, False])
label_counts = (
    aggregated_df
    .groupby(['level', 'label'])
    .size()
    .reset_index(name='count')
    .sort_values(by=['level', 'count'], ascending=[True, False])
)
display(label_counts)

In [ ]:
# Plan:
# 1. Take the features above median frequency at each level
# 2. Create a set of selected features
# 3. Us them to form all possible combinations of between 2 to max of features set
# 4. hyper tuning with featureset as hyperparameter as well

filtered = label_counts[label_counts['label'] != "friedman_mse"]
level_1 = filtered[filtered['level'] == 1]
level_2 = filtered[filtered['level'] == 2]
level_3 = filtered[filtered['level'] == 3]

selected_features = set()
for level in [level_1, level_2, level_3]:
    median_count = level['count'].median()
    features_above_median = level[level['count'] >= median_count]['label'].tolist()
    selected_features.update(features_above_median)

print("Total shortlisted features: " + str(len(selected_features)))
print(selected_features)

combinations = []
from itertools import combinations as iter_combinations
for r in range(2, len(selected_features) + 1):
    combinations.extend(iter_combinations(selected_features, r))

print(f'Total combinations: {len(combinations)}')


In [ ]:
for i in range(1500, 1510):
    print(combinations[i])

In [ ]:
aggregated_df.value_counts("label")

In [ ]:
aggregated_df

In [ ]:
def getScore(levels):
    scores = []
    score = 0
    for level in levels:
        if level == 0:
            score = 3
        elif level == 1:
            score = 2
        elif level == 2:
            score = 1
        else:
            score = 0
        scores.append(score)
    return scores

## Feature extraction

In [ ]:
label_counts_with_score = label_counts.copy()
label_counts_with_score['score'] = (label_counts_with_score['count'] * getScore(label_counts_with_score['level']))
label_counts_with_score = label_counts_with_score.drop(columns=['count', 'level'])
label_counts_with_score = label_counts_with_score.groupby(['label'])['score'].sum().reset_index()
label_counts_with_score = label_counts_with_score.sort_values(by=['score'], ascending=False)
label_counts_with_score

In [ ]:
best_model.feature_importances_
feat_importances = pd.Series(best_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
feat_importances.sort_values(ascending=False)

## Model Retrain

In [ ]:
feature_group_list = []
ranked_features = label_counts_with_score['label'].tolist()

#  Manually reset feature name
for i, feature in enumerate(ranked_features):
    if feature == "AtSteroid":
        ranked_features[i] = "AtSteroid Dose_term"
    if feature == "Intrapartum":
        ranked_features[i] = "Intrapartum Antibiotic_term"

ranked_features.remove("friedman_mse")

for i, feature in enumerate(ranked_features):
    feature_group_list.append(ranked_features[:i+1])

for i, items in enumerate(feature_group_list):
    print(f"Feature Group {i+1}: {items}")

In [ ]:
feature_group_list_result = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

for feature_group in feature_group_list:
    for model in models:
        print(f"Evaluating {model['model_name']} with feature group {feature_group}")
        feature_group_result = {}
        feature_group_list_result.append(feature_group_result)
        feature_group_result["model"] = model["model_name"]
        feature_group_result["feature list"] = feature_group

        steps = []
        steps.append(("sampler", RandomOverSampler(random_state=123)))
        steps.append(("model", model["model"]))
        
        pipeline = ImbPipeline(steps=steps)
        
        temp_X_train = X_train[feature_group]
        
        for scoring_name in scoring.keys():
            scores = cross_val_score(
                pipeline, temp_X_train, y_train, cv=cv, scoring=scoring[scoring_name]
            )
            feature_group_result[scoring_name] = np.mean(scores)

In [ ]:
feature_group_list_result_df = pd.DataFrame(feature_group_list_result)
feature_group_list_result_df

In [ ]:
feature_group_list_result_df.to_csv('feature_group_list_result_df.csv')

## Inspect Results by group average

In [ ]:
average_by_auroc = feature_group_list_result_df.copy(deep=True)
average_by_auroc["feature list"] = average_by_auroc["feature list"].apply(lambda x: str(x))
average_by_auroc = average_by_auroc\
    .drop(columns=["model", "f1", "precision", "accuracy", "balanced_accuracy", "recall"])\
    .groupby(by=["feature list"]).mean()\
    .sort_values(by=["roc_auc"], ascending=False)
average_by_auroc

In [ ]:
# highest average auroc by feature list (combining all models results))
highest_ave = ['GA', 'BW', 'MotherAge', 'RespSpt_HFOVDur', 'RespSpt_HFNCDone', 'RespSpt_CPAPDone', 'Apgar1Min', 'BirthGrowthStatus_term', 'RespSpt_HFOVDone', 'RespSpt_CPAPDur', 'BirthAdmissionTemp', 'RespSpt_ConvVentDur', 'PtRace_term']
len(highest_ave)

## Sort results directly, regardless of grouping

In [ ]:
highest_by_auroc = feature_group_list_result_df.copy(deep=True)
highest_by_auroc = highest_by_auroc.sort_values(by=["roc_auc"], ascending=False)
highest_by_auroc

In [ ]:
highest_auroc = highest_by_auroc.iloc[0]["feature list"]
len(highest_auroc)

## Try with extra tree hyper parameter tuning as it performs very well individually for the newly ranked importance metrics

In [ ]:
feature_group_list_result = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

for i, feature_group in enumerate(feature_group_list):
    progress_display = f"Progress: {'|'*(i+1) + ' '*(len(feature_group_list)-i)} {i/len(feature_group_list)*100:.1f}%"
    print(progress_display)
    print(f"Evaluating Extra Tree Classifier with feature group {feature_group}")
    feature_group_result = {}
    feature_group_list_result.append(feature_group_result)
    feature_group_result["model"] = "Extra Trees Classifier"
    feature_group_result["feature list"] = feature_group

    steps = []
    steps.append(("sampler", RandomOverSampler(random_state=123)))
    steps.append(("model", ExtraTreesClassifier(criterion="entropy",random_state=123)))
    
    param_grid = {
            "model__n_estimators": [100, 200, 500],
            "model__criterion": ["gini", "entropy", "log_loss"],
            "model__max_depth": [None, 10, 20, 50],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": ["sqrt", "log2", None]
        }
    
    pipeline = ImbPipeline(steps=steps)
    
    temp_X_train = X_train[feature_group]
    
    grid = GridSearchCV(
        pipeline,
        param_grid,
        cv=cv,
        scoring="balanced_accuracy",
        n_jobs=-1
    )    
    grid.fit(temp_X_train, y_train)
    
    feature_group_result["best_balanced_accuracy"] = grid.best_score_
    feature_group_result["best_params"] = grid.best_params_

In [ ]:
feature_group_list_result_df = pd.DataFrame(feature_group_list_result)
feature_group_list_result_df.sort_values(by=["best_balanced_accuracy"], ascending=False)

## Retrain Extra Tree with Best Hyperparameters and Best Feature Set

In [ ]:
extra_tree_best_params = {'model__criterion': 'entropy', 
    'model__max_depth': None, 
    'model__max_features': 'sqrt', 
    'model__min_samples_leaf': 2, 
    'model__min_samples_split': 2, 
    'model__n_estimators': 100
    }

extra_tree_best_params_for_all_features = {'model__criterion': 'entropy', 
    'model__max_depth': None, 
    'model__max_features': 'log2', 
    'model__min_samples_leaf': 4, 
    'model__min_samples_split': 2, 
    'model__n_estimators': 100}

Ori = {'model__criterion': 'entropy',
   'model__max_depth': None,
   'model__max_features': None,
   'model__min_samples_leaf': 4,
   'model__min_samples_split': 2,
   'model__n_estimators': 100}

best_model = ExtraTreesClassifier(criterion="entropy",
    random_state=123,
    max_depth=None,
    max_features='sqrt',
    min_samples_leaf=2,
    min_samples_split=2,
    n_estimators=100
)

best_features = ['BW', 'MotherAge', 'RespSpt_HFOVDone', 'RespSpt_HFNCDone', 'GA', 'RespSpt_CPAPDone', 'BirthAdmissionTemp', 'RespSpt_HFOVDur']

X_temp = X_train[best_features]
X_ros, y_ros = RandomOverSampler(random_state=123).fit_resample(X_temp, y_train)
# X_ros, y_ros = RandomOverSampler(random_state=123).fit_resample(X_train, y_train)
best_model.fit(X_ros, y_ros)

In [ ]:
y_pred = best_model.predict(X_test[best_features])
y_pred_proba = best_model.predict_proba(X_test[best_features])[:, 1]
res = {
    "f1" : f1_score(y_test, y_pred, average="binary", pos_label="Dead"),
    "roc_auc" : roc_auc_score(y_test, y_pred_proba),
    "precision" : precision_score(y_test, y_pred, pos_label="Dead"),
    "accuracy" : accuracy_score(y_test, y_pred),
    "balanced_accuracy" : balanced_accuracy_score(y_test, y_pred),
    "recall" : recall_score(y_test, y_pred, pos_label="Dead")
}
res

## Retrain Extra Tree with Best Hyperparameters only

In [ ]:
extra_tree_best_params = {'model__criterion': 'entropy', 
    'model__max_depth': None, 
    'model__max_features': 'sqrt', 
    'model__min_samples_leaf': 2, 
    'model__min_samples_split': 2, 
    'model__n_estimators': 100
    }

best_model = ExtraTreesClassifier(criterion="entropy",
    random_state=123,
    max_depth=None,
    max_features='sqrt',
    min_samples_leaf=2,
    min_samples_split=2,
    n_estimators=100
)
X_ros, y_ros = RandomOverSampler(random_state=123).fit_resample(X_train, y_train)
best_model.fit(X_ros, y_ros)

In [ ]:
y_pred = best_model.predict(X_test)
y_pred_proba = best_model.predict_proba(X_test)[:, 1]
res = {
    "f1" : f1_score(y_test, y_pred, average="binary", pos_label="Dead"),
    "roc_auc" : roc_auc_score(y_test, y_pred_proba),
    "precision" : precision_score(y_test, y_pred, pos_label="Dead"),
    "accuracy" : accuracy_score(y_test, y_pred),
    "balanced_accuracy" : balanced_accuracy_score(y_test, y_pred),
    "recall" : recall_score(y_test, y_pred, pos_label="Dead")
}
res

# Comparison

In [ ]:
# Initial Best Model: Gradient Boosting Classifier with RandomOverSampler
GBC_res = {'f1': 0.8125,
 'roc_auc': 0.9732558139534884,
 'precision': 0.8125,
 'accuracy': 0.974025974025974,
 'balanced_accuracy': 0.8992732558139536,
 'recall': 0.8125}

# Final Best Model: Extra Trees Classifier with RandomOverSampler and selected features
ETC_res = {'f1': 0.8125,
 'roc_auc': 0.9738372093023255,
 'precision': 0.8125,
 'accuracy': 0.974025974025974,
 'balanced_accuracy': 0.8992732558139536,
 'recall': 0.8125}

# Final Best Model: Extra Trees Classifier with RandomOverSampler and all features
ETC_ff_res = {'f1': 0.896551724137931,
 'roc_auc': 0.9938953488372093,
 'precision': 1.0,
 'accuracy': 0.987012987012987,
 'balanced_accuracy': 0.90625,
 'recall': 0.8125}

# Test with highest average Feature Group

In [ ]:
feature_set = ['BW', 'MotherAge', 'RespSpt_HFOVDone', 'RespSpt_HFNCDone', 'GA', 'RespSpt_CPAPDone', 'BirthAdmissionTemp', 'RespSpt_HFOVDur']

In [ ]:
best_hyperparameters_by_models = []
X_temp = X_train[feature_set]

for model in models:
    print(f"Evaluating {model['model_name']} with hyperparameter tuning")
    best_hyperparameters = {}
    best_hyperparameters_by_models.append(best_hyperparameters)
    best_hyperparameters["model"] = model["model_name"]
    
    pipeline = ImbPipeline([
        ("sampler", RandomOverSampler(random_state=123)),
        ("model", model["model"])
    ])

    param_grid = model["hyperparameters"]

    grid = GridSearchCV(
        pipeline,
        param_grid,
        cv=cv,
        scoring="balanced_accuracy",
        n_jobs=-1
    )    
    grid.fit(X_temp, y_train)
    
    best_hyperparameters["best_balanced_accuracy"] = grid.best_score_
    best_hyperparameters["best_params"] = grid.best_params_
    print(f"{model['model_name']}: best balanced_accuracy {grid.best_score_:.3f} with {grid.best_params_}")

In [ ]:
best_hyperparameters_by_models

In [ ]:
best_hyperparameters_by_models_df = pd.DataFrame(best_hyperparameters_by_models)
best_hyperparameters_by_models_df

# New Approach

1. Conventionaly way of HPT and get best model
2. Analyse best model and learn how it prioritise features
3. Split into feature groups and tune hyper parameters with different feature groups
4. Apply best hyper parameters out of step 3 (hypothesis: performance will improve)

- We need to compare with RFE and see the trend
- Model transparency:
  - Feature selection
  - Determine best model
  - Try to explain best model
  - Get clinicians' feedback